# ЛР №1: настройка окружения и подготовка выборок

Ноутбук со всем ходом выполнения: проверка окружения, выборки 1-4, нормализация, визуализация, финальная проверка (раздел 9 методички).

## Проверка окружения

In [4]:
import sys, pandas, numpy, matplotlib, sklearn, drawdata

print(sys.version)
print("executable:", sys.executable)
print("pandas", pandas.__version__, "| numpy", numpy.__version__)
print("sklearn", sklearn.__version__, "| matplotlib", matplotlib.__version__)

ModuleNotFoundError: No module named 'pandas'

## Общая подготовка (раздел 4)

Пути и две вспомогательные функции: нормализация признаков и визуализация разброса с раскраской по метке.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
from sklearn.preprocessing import MinMaxScaler
from pandas.plotting import scatter_matrix, parallel_coordinates
from matplotlib.lines import Line2D

DATA = Path("../data")
FIG = Path("../figures")
DATA.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

palette = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]


def normalize_features(df, features):
    """Приводит перечисленные признаки к [0, 1], не трогая остальные колонки (метку в том числе)."""
    df = df.copy()
    scaler = MinMaxScaler()
    df[features] = scaler.fit_transform(df[features])
    return df


def plot_clusters_2d(df, x, y, label_col, title, filename):
    """Разброс x-y с раскраской по метке, легендой и сохранением в figures/."""
    fig, ax = plt.subplots(figsize=(6, 5))
    for value, group in df.groupby(label_col):
        ax.scatter(group[x], group[y], s=12, label=f"кластер {value}")
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    ax.legend()
    fig.savefig(FIG / filename, dpi=120, bbox_inches="tight")
    plt.show()
    return fig


def scatter_matrix_with_legend(df, features, cluster_col, filename, legend_labels=None):
    """scatter_matrix + легенда по кластерам (сам scatter_matrix легенду не строит)."""
    codes, uniques = pd.factorize(df[cluster_col]) if df[cluster_col].dtype == object else (df[cluster_col].to_numpy(), sorted(df[cluster_col].unique()))
    axes = scatter_matrix(
        df[features],
        c=[palette[c % len(palette)] for c in codes],
        s=12, alpha=0.8, diagonal="hist", figsize=(8, 8),
    )
    fig = axes[0, 0].get_figure()
    labels = legend_labels if legend_labels is not None else [f"кластер {u}" for u in sorted(set(codes))]
    fig.legend(
        handles=[Line2D([], [], marker="o", linestyle="", color=palette[i % len(palette)], label=lab)
                 for i, lab in enumerate(labels)],
        loc="upper right",
    )
    fig.savefig(FIG / filename, dpi=120, bbox_inches="tight")
    plt.show()
    return fig

ModuleNotFoundError: No module named 'numpy'

## Выборка 1: нарисованная мышью (раздел 5)

**Эту часть нельзя автоматизировать** — нужно самостоятельно нарисовать мышью 3-5 линейно-разделимых кластеров, всего от 1000 точек, каждый кластер своим цветом. Ниже — рабочий код по шагам 5.1 методички.

In [ ]:
from drawdata import ScatterWidget

# Нарисуйте мышью 3-5 не соприкасающихся кластеров, ~1000+ точек суммарно.
# Параметры width, height, brushsize можно передать в конструктор, если нужно
# менять размер полотна или кисти: ScatterWidget(width=700, height=500, brushsize=15)
widget = ScatterWidget()
widget

In [ ]:
# Выполнить ПОСЛЕ того как нарисовано (в отдельной ячейке — так по методичке,
# иначе data_as_pandas вернёт пустую таблицу).
raw1 = widget.data_as_pandas
print(raw1.shape)
raw1.head()

In [ ]:
sample1 = raw1.copy()
sample1["cluster"], _ = pd.factorize(sample1["color"])
sample1 = sample1[["x", "y", "cluster"]]

# Виджет отдаёт экранные координаты (ось y растёт вниз). Если график выглядит
# перевёрнутым по вертикали, раскомментировать строку ниже:
# sample1["y"] = 1 - sample1["y"]

sample1 = normalize_features(sample1, ["x", "y"])
sample1.to_csv(DATA / "sample1.csv", index=False)

plot_clusters_2d(sample1, "x", "y", "cluster", "Выборка 1: нарисованные кластеры", "sample1.png")

Проверка (опционально, раздел 5.2):

In [ ]:
print(sample1.shape)                                    # строк от 1000, колонок 3
print(sample1["cluster"].value_counts().sort_index())   # 3-5 кластеров, размеры соизмеримы
print(sample1[["x", "y"]].agg(["min", "max"]))           # 0.0 и 1.0

## Выборка 2: сгенерированная (раздел 6)

Функция генерации по контракту раздела 6.1, подбор диапазона σ по фактическому расстоянию между центрами (раздел 6.3), проверка степени пересечения.

In [ ]:
def generate_sample(K, N_k, sigma_range, seed=None, center_range=(0.0, 100.0)):
    """
    K            - количество кластеров
    N_k          - количество точек в каждом кластере
    sigma_range  - кортеж (sigma_1, sigma_2), диапазон дисперсии
    seed         - зерно генератора для воспроизводимости
    center_range - границы, в которых случайно выбираются центры
    Возвращает (DataFrame с колонками x, y, z, cluster; centers; sigmas)
    """
    rng = np.random.default_rng(seed)
    centers = rng.uniform(center_range[0], center_range[1], size=(K, 3))
    sigmas = rng.uniform(sigma_range[0], sigma_range[1], size=K)
    parts = []
    for k in range(K):
        pts = rng.normal(centers[k], sigmas[k], size=(N_k, 3))
        parts.append(pd.DataFrame(pts, columns=["x", "y", "z"]).assign(cluster=k))
    df = pd.concat(parts, ignore_index=True)
    return df, centers, sigmas


def suggest_sigma_range(K, seed, center_range=(0.0, 100.0), low=0.25, high=0.33):
    """Подбирает диапазон sigma по фактическому d_min между центрами при том же seed.
    Использует тот же порядок обращений к rng, что и generate_sample (сначала центры),
    поэтому при одинаковом seed центры совпадут в обеих функциях."""
    rng = np.random.default_rng(seed)
    centers = rng.uniform(center_range[0], center_range[1], size=(K, 3))
    d_min = min(np.linalg.norm(centers[i] - centers[j])
                for i, j in combinations(range(K), 2))
    return (low * d_min, high * d_min), d_min


def cluster_report(df, centers, sigmas, cols=("x", "y", "z")):
    X = df[list(cols)].to_numpy()
    d = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
    overlap = 100.0 * (d.argmin(axis=1) != df["cluster"].to_numpy()).mean()
    pairs = list(combinations(range(len(centers)), 2))
    d_min = min(np.linalg.norm(centers[i] - centers[j]) for i, j in pairs)
    sep = min(np.linalg.norm(centers[i] - centers[j]) / (sigmas[i] + sigmas[j])
              for i, j in pairs)
    print(f"мин. расстояние между центрами: {d_min:.1f}")
    print(f"коэффициент раздельности: {sep:.2f} (целевой диапазон 1.5-2.0)")
    print(f"пересечение кластеров: {overlap:.1f}% (целевой диапазон 2-8%)")
    return overlap, sep

In [ ]:
# Параметры генерации — зафиксированы здесь же, этого достаточно для воспроизведения.
K2, N_k2, SEED2 = 4, 300, 2026

sigma_range2, d_min2 = suggest_sigma_range(K2, seed=SEED2)
raw2, centers2, sigmas2 = generate_sample(K=K2, N_k=N_k2, sigma_range=sigma_range2, seed=SEED2)

cluster_report(raw2, centers2, sigmas2)

sample2 = normalize_features(raw2, ["x", "y", "z"])
sample2.to_csv(DATA / "sample2.csv", index=False)

# Дублируем параметры отдельным файлом — удобно, если придётся подбирать заново.
pd.DataFrame({
    "K": [K2], "N_k": [N_k2], "seed": [SEED2],
    "sigma_low": [sigma_range2[0]], "sigma_high": [sigma_range2[1]],
}).to_csv(DATA / "sample2_params.csv", index=False)

In [ ]:
# Тривимірне розсіювання
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection="3d")
for value, group in sample2.groupby("cluster"):
    ax.scatter(group["x"], group["y"], group["z"], s=8, label=f"кластер {value}")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.legend()
fig.savefig(FIG / "sample2_3d.png", dpi=120, bbox_inches="tight")
plt.show()

# Матрица рассеяния 3x3 — честнее показывает пересечение, чем один ракурс 3D
scatter_matrix_with_legend(sample2, ["x", "y", "z"], "cluster", "sample2_matrix.png")

Проверка структуры (опционально, раздел 6.2):

In [ ]:
assert raw2.shape == (K2 * N_k2, 4), raw2.shape
assert list(raw2.columns) == ["x", "y", "z", "cluster"]
assert raw2.cluster.nunique() == K2
assert raw2.groupby("cluster").size().nunique() == 1
print("K x N_k =", K2 * N_k2, "-> >= 1000:", K2 * N_k2 >= 1000)

## Выборка 3: ирисы Фишера (раздел 7)

`data/iris.csv` должен быть скопирован в проект заранее (см. конец раздела 2 методички).

In [ ]:
iris = pd.read_csv(DATA / "iris.csv")
print(iris.shape)
print("NaN всего:", iris.isna().sum().sum())
iris.head()

In [ ]:
sample3 = iris.copy()
sample3["cluster"], species_labels = pd.factorize(sample3["Species"])  # текстовая метка сохраняется рядом

features3 = ["Sepal_length", "Sepal_width", "Petal_length", "Petal_width"]
sample3 = normalize_features(sample3, features3)
sample3.to_csv(DATA / "sample3.csv", index=False)

print(sample3.groupby("Species")["cluster"].first())

In [ ]:
legend_labels3 = [f"кластер {i} ({sp})" for i, sp in enumerate(species_labels)]
scatter_matrix_with_legend(sample3, features3, "cluster", "sample3_matrix.png", legend_labels3)

Проверка (опционально, раздел 7.2):

In [ ]:
print(iris.shape)                              # (150, 5)
print(iris.isna().sum().sum())                 # 0
print(sample3.groupby("Species")["cluster"].first())  # три разных номера

## Выборка 4: реальные данные (раздел 8)

Ниже — пример на наборе `wine-quality-red` из методички. **Замените набор на свой** (условие уникальности в подгруппе) — сохранится вся структура кода, поменяются только имя набора, `features4` и, возможно, название колонки метки.

In [ ]:
from sklearn.datasets import fetch_openml

wine = fetch_openml(name="wine-quality-red", version=1, as_frame=True)
df4 = wine.frame.copy()
print(df4.shape)
print(df4.dtypes)

Обзор перед обработкой (опционально, раздел 8.1):

In [ ]:
print(df4.isna().sum())
df4.describe().T

In [ ]:
# Пропущенные значения (раздел 8.2): сначала выбросить сильно неполные строки,
# затем заполнить единичные пропуски медианой (устойчивее к ещё не убранным выбросам).
rows_before = df4.shape[0]
df4 = df4.dropna(thresh=int(0.7 * df4.shape[1]))

num_cols = df4.select_dtypes("number").columns.tolist()
df4[num_cols] = df4[num_cols].fillna(df4[num_cols].median())

print(f"строк было {rows_before}, стало {df4.shape[0]}")

In [ ]:
# Выбросы, правило 1.5*IQR (раздел 8.3). Метка (class) и константные признаки исключены.
label_col = "class"
features4 = [c for c in num_cols if c != label_col and df4[c].std() > 0]

Q1 = df4[features4].quantile(0.25)
Q3 = df4[features4].quantile(0.75)
IQR = Q3 - Q1
k_iqr = 1.5
lower = Q1 - k_iqr * IQR
upper = Q3 + k_iqr * IQR

would_drop = ((df4[features4] < lower) | (df4[features4] > upper)).any(axis=1)
print(f"удаление затронуло бы {would_drop.sum()} из {len(df4)} строк "
      f"({100 * would_drop.mean():.1f}%)")

# Если потери > 10% ориентира — обрезаем значения вместо удаления строк.
df4_clipped = df4.copy()
df4_clipped[features4] = df4_clipped[features4].clip(lower=lower, upper=upper, axis=1)
changed_rows = (df4[features4] != df4_clipped[features4]).any(axis=1).sum()

if would_drop.mean() > 0.10:
    print(f"обрезание: строк {len(df4_clipped)} (было {len(df4)}), "
          f"подрезаны значения в {changed_rows} строках "
          f"({100 * changed_rows / len(df4):.1f}%)")
    df4 = df4_clipped
else:
    print(f"удаление: строк было {len(df4)}, стало {len(df4) - would_drop.sum()}")
    df4 = df4.loc[~would_drop]

In [ ]:
# Теплокарта корреляций — инструмент отбора признаков, не картина кластеров (раздел 4.1)
corr = df4[features4].corr()
fig4, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(features4)), features4, rotation=90, fontsize=8)
ax.set_yticks(range(len(features4)), features4, fontsize=8)
fig4.colorbar(im, ax=ax, shrink=0.8)
fig4.savefig(FIG / "sample4_corr.png", dpi=120, bbox_inches="tight")
plt.show()

# Отбор 4-5 признаков для матрицы: по std, отбрасывая те, что коррелируют > 0.6 с уже отобранными
ranked = df4[features4].std().sort_values(ascending=False).index.tolist()
selected = []
for f in ranked:
    if all(abs(corr.loc[f, s]) <= 0.6 for s in selected):
        selected.append(f)
    if len(selected) == 5:
        break
print("отобранные признаки для матрицы рассеяния:", selected)

In [ ]:
sample4 = normalize_features(df4, features4)
sample4.to_csv(DATA / "sample4.csv", index=False)
print(sample4.shape)

legend_labels4 = [f"класс {c}" for c in sorted(sample4[label_col].unique())]
scatter_matrix_with_legend(sample4, selected, label_col, "sample4_matrix.png", legend_labels4)

In [ ]:
# Паралельні координати по подвыборке (1599 строк слились бы в сплошную заливку)
sub = sample4.sample(min(200, len(sample4)), random_state=0)
fig6, ax = plt.subplots(figsize=(10, 4))
parallel_coordinates(
    sub[features4 + [label_col]], label_col, ax=ax,
    color=palette[:sub[label_col].nunique()], alpha=0.3,
)
ax.set_ylim(0, 1)
ax.tick_params(axis="x", labelrotation=90, labelsize=8)
fig6.savefig(FIG / "sample4_parallel.png", dpi=120, bbox_inches="tight")
plt.show()

Проверка (опционально):

In [ ]:
print(f"строк: {len(sample4)} (>= 1000: {len(sample4) >= 1000})")
print(f"признаков: {len(features4)} (>= 5: {len(features4) >= 5})")
print(f"метка '{label_col}' присутствует:", label_col in sample4.columns)

## Финальная проверка (раздел 9, опционально)

In [ ]:
SPEC = {
    "sample1": ["x", "y"],
    "sample2": ["x", "y", "z"],
    "sample3": features3,
    "sample4": features4,
}

for name, feats in SPEC.items():
    df = pd.read_csv(DATA / f"{name}.csv")
    f = df[feats]
    print(f"{name}: {df.shape[0]} строк, признаков {len(feats)}, "
          f"NaN={int(df.isna().sum().sum())}, "
          f"min={f.min().min():.3f}, max={f.max().max():.3f}")